# gffutils smoke test

Exercises the primary API surface of `gffutils` 0.14: building a SQLite-backed
feature DB from a small in-memory GFF3 snippet, querying features by type /
region / parent-child relationships, and emitting features back as text.

In [ ]:
# Example 1: build an in-memory FeatureDB from a small GFF3 snippet.
import gffutils

GFF3 = """\
##gff-version 3
chr1\ttest\tgene\t1000\t9000\t.\t+\t.\tID=gene1;Name=GENE1
chr1\ttest\tmRNA\t1050\t9000\t.\t+\t.\tID=mrna1;Parent=gene1
chr1\ttest\texon\t1050\t1500\t.\t+\t.\tID=exon1;Parent=mrna1
chr1\ttest\texon\t3000\t3500\t.\t+\t.\tID=exon2;Parent=mrna1
chr1\ttest\texon\t8000\t9000\t.\t+\t.\tID=exon3;Parent=mrna1
"""

db = gffutils.create_db(GFF3, dbfn=':memory:', from_string=True, force=True, keep_order=True)
print(f"db type: {type(db).__name__}")
print(f"feature count: {sum(1 for _ in db.all_features())}")
assert isinstance(db, gffutils.FeatureDB)

In [ ]:
# Example 2: lookup by ID + featuretypes() + count_features_of_type().
import gffutils

GFF3 = """\
##gff-version 3
chr1\ttest\tgene\t1000\t9000\t.\t+\t.\tID=gene1
chr1\ttest\tmRNA\t1050\t9000\t.\t+\t.\tID=mrna1;Parent=gene1
chr1\ttest\texon\t1050\t1500\t.\t+\t.\tID=exon1;Parent=mrna1
chr1\ttest\texon\t3000\t3500\t.\t+\t.\tID=exon2;Parent=mrna1
"""
db = gffutils.create_db(GFF3, dbfn=':memory:', from_string=True, force=True)

gene = db['gene1']
print(f"gene1: seqid={gene.seqid} start={gene.start} end={gene.end} strand={gene.strand}")

types = sorted(db.featuretypes())
print(f"featuretypes: {types}")
exon_count = db.count_features_of_type('exon')
print(f"exon count: {exon_count}")
assert types == ['exon', 'gene', 'mRNA']
assert exon_count == 2

In [ ]:
# Example 3: parent/child traversal via db.children() and db.parents().
import gffutils

GFF3 = """\
##gff-version 3
chr1\ttest\tgene\t1000\t9000\t.\t+\t.\tID=gene1
chr1\ttest\tmRNA\t1050\t9000\t.\t+\t.\tID=mrna1;Parent=gene1
chr1\ttest\texon\t1050\t1500\t.\t+\t.\tID=exon1;Parent=mrna1
chr1\ttest\texon\t3000\t3500\t.\t+\t.\tID=exon2;Parent=mrna1
chr1\ttest\texon\t8000\t9000\t.\t+\t.\tID=exon3;Parent=mrna1
"""
db = gffutils.create_db(GFF3, dbfn=':memory:', from_string=True, force=True)

exons = list(db.children('mrna1', featuretype='exon', order_by='start'))
exon_ids = [e.id for e in exons]
print(f"mrna1 exons (ordered): {exon_ids}")

parents = list(db.parents('exon2'))
parent_ids = sorted(p.id for p in parents)
print(f"exon2 ancestors: {parent_ids}")

assert exon_ids == ['exon1', 'exon2', 'exon3']
assert parent_ids == ['gene1', 'mrna1']

In [ ]:
# Example 4: region-based queries via db.region().
import gffutils

GFF3 = """\
##gff-version 3
chr1\ttest\tgene\t1000\t2000\t.\t+\t.\tID=geneA
chr1\ttest\tgene\t5000\t7000\t.\t+\t.\tID=geneB
chr2\ttest\tgene\t100\t900\t.\t-\t.\tID=geneC
"""
db = gffutils.create_db(GFF3, dbfn=':memory:', from_string=True, force=True)

hits = list(db.region(seqid='chr1', start=1500, end=6000, featuretype='gene'))
hit_ids = sorted(h.id for h in hits)
print(f"chr1:1500-6000 genes: {hit_ids}")

chr2_hits = list(db.region(seqid='chr2'))
print(f"chr2 features: {[h.id for h in chr2_hits]}")

assert hit_ids == ['geneA', 'geneB']
assert [h.id for h in chr2_hits] == ['geneC']

In [ ]:
# Example 5: Feature attribute access + GFF3 round-trip via str(feature).
import gffutils

GFF3 = """\
##gff-version 3
chr1\ttest\tgene\t1000\t9000\t.\t+\t.\tID=gene1;Name=GENE1;biotype=protein_coding
"""
db = gffutils.create_db(GFF3, dbfn=':memory:', from_string=True, force=True)

gene = db['gene1']
print(f"attributes keys: {sorted(gene.attributes.keys())}")
print(f"Name: {gene.attributes['Name']}")
print(f"biotype: {gene.attributes['biotype']}")
print(f"length: {len(gene)}")

rendered = str(gene)
print(f"rendered: {rendered}")

assert gene.attributes['Name'] == ['GENE1']
assert gene.attributes['biotype'] == ['protein_coding']
assert len(gene) == 9000 - 1000 + 1
assert 'ID=gene1' in rendered and 'Name=GENE1' in rendered
print("\n✅ gffutils smoke test passed")